# 03 วิเคราะห์ข้อมูลและสร้างแบบจำลอง
ตัวเลขและรูปทุกค่าในบทที่ 3 มาจาก notebook นี้ ผลลัพธ์ตารางอยู่ใน `results/` และรูปอยู่ใน `figures/`

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
FIG = ROOT / "figures" / "regenerated"   # รูปที่ notebook สร้างใหม่ (รูปในเล่มอยู่ที่ figures/)
RES = ROOT / "results"
for p in (RAW, PROC, FIG, RES):
    p.mkdir(parents=True, exist_ok=True)

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

plt.rcParams.update({"figure.dpi": 150, "axes.grid": True, "grid.alpha": 0.3})
df = pd.read_csv(PROC / "BangPaKong_clean.csv", parse_dates=["DateTime"])
y = df["WaterLevel"]

## 3.1 สถิติเชิงพรรณนา

In [3]:
desc = pd.DataFrame({"n": [len(y)], "mean": [y.mean()], "sd": [y.std()],
                     "min": [y.min()], "median": [y.median()], "max": [y.max()]}).round(2)
desc.to_csv(RES / "table_desc.csv", index=False)
print(desc.to_string(index=False))
print("จำนวนจุดที่ระดับน้ำ = 0:", int((y == 0).sum()))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["DateTime"], y, lw=0.3)
ax.set_xlabel("Date"); ax.set_ylabel("Water level (m)")
fig.tight_layout(); fig.savefig(FIG / "fig_series.png"); plt.close(fig)

     n  mean   sd  min  median  max
352684  2.83 0.79  0.0    2.95 5.78
จำนวนจุดที่ระดับน้ำ = 0: 38


## 3.2 ความแปรผันตามฤดูกาล (คำถามที่ 1)

In [4]:
g = df.groupby("Month")["WaterLevel"].agg(["mean", "std"]).round(2)
g.to_csv(RES / "table_monthly.csv")
print(g.T.to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].bar(g.index, g["mean"]); ax[0].set_xlabel("Month"); ax[0].set_ylabel("Mean water level (m)")
ax[0].set_xticks(range(1, 13)); ax[0].set_title("(a)")
ax[1].boxplot([df.loc[df["Month"] == k, "WaterLevel"] for k in range(1, 13)], tick_labels=range(1, 13))
ax[1].set_xlabel("Month"); ax[1].set_ylabel("Water level (m)"); ax[1].set_title("(b)")
fig.tight_layout(); fig.savefig(FIG / "fig_month.png"); plt.close(fig)

Month    1     2     3     4     5     6     7     8     9     10    11    12
mean   2.90  2.92  2.88  2.80  2.73  2.66  2.69  2.73  2.79  2.90  3.01  2.96
std    0.78  0.75  0.72  0.74  0.78  0.80  0.81  0.79  0.77  0.79  0.89  0.79


## 3.3 การแจกแจงและค่าผิดปกติ (คำถามที่ 2)

In [5]:
q1, q3 = y.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_lo, n_hi = int((y < lo).sum()), int((y > hi).sum())
out = pd.DataFrame({"Q1": [q1], "Q3": [q3], "IQR": [iqr], "lower": [lo], "upper": [hi],
                    "n_low": [n_lo], "n_high": [n_hi], "n_total": [n_lo + n_hi],
                    "pct": [100 * (n_lo + n_hi) / len(y)], "skew": [y.skew()]}).round(2)
out.to_csv(RES / "table_outlier.csv", index=False)
print(out.T.to_string(header=False))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(y, bins=np.arange(0, 6.01, 0.25), edgecolor="white")
ax[0].set_xlabel("Water level (m)"); ax[0].set_ylabel("Frequency"); ax[0].set_title("(a)")
ax[1].boxplot(y, vert=False); ax[1].set_xlabel("Water level (m)"); ax[1].set_title("(b)")
fig.tight_layout(); fig.savefig(FIG / "fig_hist_box.png"); plt.close(fig)

Q1         2.24
Q3         3.46
IQR        1.22
lower      0.41
upper      5.29
n_low     38.00
n_high   564.00
n_total  602.00
pct        0.17
skew      -0.33


## 3.4 แนวโน้มรายปี (คำถามที่ 3)

In [6]:
a = df.groupby("Year")["WaterLevel"].agg(n="size", mean="mean", max="max").round(2)
a.to_csv(RES / "table_annual.csv")
print(a.to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(a.index, a["mean"], marker="o", label="Mean")
ax.plot(a.index, a["max"], marker="s", label="Maximum")
ax.set_xlabel("Year"); ax.set_ylabel("Water level (m)"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "fig_annual.png"); plt.close(fig)

          n  mean   max
Year                   
2019  25249  2.81  4.38
2020  49408  2.83  4.52
2021  40049  2.83  4.54
2022  52890  2.88  5.78
2023  53038  2.79  4.61
2024  53523  2.82  4.44
2025  53109  2.84  4.86
2026  25418  2.84  4.41


## 3.5 ทำนายระดับน้ำเฉลี่ยรายเดือน (คำถามที่ 4)
ข้อมูลรายเดือน แบ่ง 80:20 ตามลำดับเวลา แล้วประเมิน **ทั้งชุดฝึกและชุดทดสอบ** ของทุกแบบจำลอง

In [7]:
m = df.set_index("DateTime")["WaterLevel"].resample("ME").mean().dropna().to_frame("WaterLevel")
m["Year"], m["Month"] = m.index.year, m.index.month
m["t"] = (m["Year"] - 2019) * 12 + m["Month"] - 5            # months since May 2019
for k in (12, 6):
    m[f"sin_{k}"] = np.sin(2 * np.pi * m["Month"] / k)
    m[f"cos_{k}"] = np.cos(2 * np.pi * m["Month"] / k)

FEATURES = ["t", "Month", "Year", "sin_12", "cos_12", "sin_6", "cos_6"]
SEASONAL = ["sin_12", "cos_12", "sin_6", "cos_6"]

n_train = round(len(m) * 0.8)                                # 80:20 split in time order
train, test = m.iloc[:n_train], m.iloc[n_train:]
print(f"เดือนทั้งหมด {len(m)} | ชุดฝึก {len(train)} ({train.index[0]:%Y-%m}..{train.index[-1]:%Y-%m})"
      f" | ชุดทดสอบ {len(test)} ({test.index[0]:%Y-%m}..{test.index[-1]:%Y-%m})")

เดือนทั้งหมด 84 | ชุดฝึก 67 (2019-05..2025-01) | ชุดทดสอบ 17 (2025-02..2026-06)


In [8]:
def metrics(y_true, y_pred):
    return {"R2": r2_score(y_true, y_pred),
            "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
            "MAE": mean_absolute_error(y_true, y_pred)}

ytr, yte = train["WaterLevel"], test["WaterLevel"]

rf = RandomForestRegressor(n_estimators=100, random_state=42).fit(train[FEATURES], ytr)
lin = LinearRegression().fit(train[SEASONAL], ytr)
month_mean = train.groupby("Month")["WaterLevel"].mean()
rf_t = RandomForestRegressor(n_estimators=100, random_state=42).fit(train[["t"]], ytr)
grand = ytr.mean()

# (ชุดฝึก, ชุดทดสอบ) ของแต่ละแบบจำลอง  ทุกแบบจำลองฝึกด้วยชุดฝึกเท่านั้น
preds = {
    "Random Forest (ตัวแปรทั้ง 7)":       (rf.predict(train[FEATURES]), rf.predict(test[FEATURES])),
    "ถดถอยเชิงเส้น (ตัวแปรเชิงฤดูกาล)":    (lin.predict(train[SEASONAL]), lin.predict(test[SEASONAL])),
    "ค่าเฉลี่ยรายเดือน":                   (train["Month"].map(month_mean).values, test["Month"].map(month_mean).values),
    "Random Forest (เฉพาะ t)":            (rf_t.predict(train[["t"]]), rf_t.predict(test[["t"]])),
    "ค่าเฉลี่ยรวม":                        (np.full(len(train), grand), np.full(len(test), grand)),
}

rows = []
for name, (p_tr, p_te) in preds.items():
    rows.append({"model": name, "set": "ชุดฝึก", **metrics(ytr, p_tr)})
    rows.append({"model": name, "set": "ชุดทดสอบ", **metrics(yte, p_te)})
res = pd.DataFrame(rows)

# เรียงแบบจำลองตาม R2 ของชุดทดสอบจากมากไปน้อย (ชุดฝึกอยู่ก่อนชุดทดสอบเสมอ)
order = (res[res["set"] == "ชุดทดสอบ"].sort_values("R2", ascending=False)["model"].tolist())
res["model"] = pd.Categorical(res["model"], categories=order, ordered=True)
res["set"] = pd.Categorical(res["set"], categories=["ชุดฝึก", "ชุดทดสอบ"], ordered=True)
res = res.sort_values(["model", "set"]).reset_index(drop=True)
res.round(3).to_csv(RES / "model_comparison.csv", index=False)
res.round(3)

,model,set,R2,RMSE,MAE
0,Random Forest (ตัวแปรทั้ง 7),ชุดฝึก,0.936,0.035,0.024
1,Random Forest (ตัวแปรทั้ง 7),ชุดทดสอบ,0.844,0.038,0.030
2,ถดถอยเชิงเส้น (ตัวแปรเชิงฤดูกาล),ชุดฝึก,0.672,0.080,0.049
3,ถดถอยเชิงเส้น (ตัวแปรเชิงฤดูกาล),ชุดทดสอบ,0.842,0.039,0.031
4,ค่าเฉลี่ยรายเดือน,ชุดฝึก,0.703,0.076,0.046
5,ค่าเฉลี่ยรายเดือน,ชุดทดสอบ,0.826,0.041,0.033
6,ค่าเฉลี่ยรวม,ชุดฝึก,0.000,0.139,0.104
7,ค่าเฉลี่ยรวม,ชุดทดสอบ,-0.000,0.097,0.087
8,Random Forest (เฉพาะ t),ชุดฝึก,0.932,0.036,0.023
9,Random Forest (เฉพาะ t),ชุดทดสอบ,-0.329,0.112,0.090


In [9]:
def fmt(v):
    s = f"{abs(v):.3f}"
    return f"$-${s}" if round(v, 3) < 0 else s

lines = []
for name in order:
    a_, b_ = (res[(res["model"] == name) & (res["set"] == s)].iloc[0] for s in ("ชุดฝึก", "ชุดทดสอบ"))
    nm = name.replace("เฉพาะ t", "เฉพาะ \\texttt{t}")
    lines.append(f"{nm} & ชุดฝึก & {fmt(a_.R2)} & {fmt(a_.RMSE)} & {fmt(a_.MAE)} \\\\ \\cline{{2-5}}")
    lines.append(f" & ชุดทดสอบ & {fmt(b_.R2)} & {fmt(b_.RMSE)} & {fmt(b_.MAE)} \\\\ \\hline")
(RES / "table_3_5_rows.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
print((RES / "table_3_5_rows.tex").read_text(encoding="utf-8"))

Random Forest (ตัวแปรทั้ง 7) & ชุดฝึก & 0.936 & 0.035 & 0.024 \\ \cline{2-5}
 & ชุดทดสอบ & 0.844 & 0.038 & 0.030 \\ \hline
ถดถอยเชิงเส้น (ตัวแปรเชิงฤดูกาล) & ชุดฝึก & 0.672 & 0.080 & 0.049 \\ \cline{2-5}
 & ชุดทดสอบ & 0.842 & 0.039 & 0.031 \\ \hline
ค่าเฉลี่ยรายเดือน & ชุดฝึก & 0.703 & 0.076 & 0.046 \\ \cline{2-5}
 & ชุดทดสอบ & 0.826 & 0.041 & 0.033 \\ \hline
ค่าเฉลี่ยรวม & ชุดฝึก & 0.000 & 0.139 & 0.104 \\ \cline{2-5}
 & ชุดทดสอบ & 0.000 & 0.097 & 0.087 \\ \hline
Random Forest (เฉพาะ \texttt{t}) & ชุดฝึก & 0.932 & 0.036 & 0.023 \\ \cline{2-5}
 & ชุดทดสอบ & $-$0.329 & 0.112 & 0.090 \\ \hline



In [10]:
# ตัวเลขประกอบข้อความในเล่ม: ฝึกด้วยข้อมูลทั้ง 84 เดือน และความสำคัญของตัวแปร
rf_all = RandomForestRegressor(n_estimators=100, random_state=42).fit(m[FEATURES], m["WaterLevel"])
print("R2 ฝึกและวัดบนข้อมูลทั้งหมด:", round(r2_score(m["WaterLevel"], rf_all.predict(m[FEATURES])), 3))
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
print(imp.round(3).sort_values(ascending=False).to_string())

pred_rf = rf.predict(m[FEATURES])
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(m.index, m["WaterLevel"], label="Actual")
ax.plot(m.index, pred_rf, "--", label="Random Forest")
ax.axvspan(test.index[0], test.index[-1], alpha=0.15, label="Test set")
ax.set_xlabel("Month"); ax.set_ylabel("Mean water level (m)"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "fig_rf_fit.png"); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
pte = preds["Random Forest (ตัวแปรทั้ง 7)"][1]
ax[0].scatter(yte, pte)
lim = [min(yte.min(), pte.min()), max(yte.max(), pte.max())]
ax[0].plot(lim, lim, "--", c="k"); ax[0].set_xlabel("Actual (m)"); ax[0].set_ylabel("Predicted (m)"); ax[0].set_title("(a)")
ax[1].barh(imp.index, imp.values); ax[1].set_xlabel("Feature importance"); ax[1].set_title("(b)")
fig.tight_layout(); fig.savefig(FIG / "fig_rf_scatter_imp.png"); plt.close(fig)

R2 ฝึกและวัดบนข้อมูลทั้งหมด: 0.943
cos_12    0.569
t         0.166
Month     0.101
Year      0.070
sin_12    0.047
cos_6     0.034
sin_6     0.013
